In [2]:
import rioxarray
import numpy as np
topography = rioxarray.open_rasterio("/Users/jkingslake/Documents/science/meltwater_routing/BFRN_meltwater/python/notebooks/rema_subsets/dem_bigger_3.tif", chunks={}).squeeze().load()
#add ocean cells around the edge of the DEM

coarsen_factor = 2
topography =topography.fillna(0).values

topography = topography[::coarsen_factor, ::coarsen_factor]
topography[0:1,:] = 0.0 
topography[-1:,:] = 0.0
topography[:,0:1] = 0.0
topography[:,-1:] = 0.0


In [3]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OMP_DYNAMIC"] = "FALSE"

import numpy as np
print("finite:", np.isfinite(topography).all())
print("nan count:", np.isnan(topography).sum())
print("min/max:", float(np.nanmin(topography)), float(np.nanmax(topography)))

ocean_level = 0.0  # your value
edge = np.r_[topography[0,:], topography[-1,:], topography[:,0], topography[:,-1]]
print("edge<=ocean:", int((edge <= ocean_level).sum()))

finite: True
nan count: 0
min/max: 0.0 800.0774536132812
edge<=ocean: 5120


In [4]:
import sys
from pathlib import Path

REPO_ROOT = Path("/Users/jkingslake/Documents/science/meltwater_routing/fsm_pywrapper/fillspillmerge")


sys.path.insert(0, str(REPO_ROOT))
from fillspillmerge import fill_spill_merge
_, dh = fill_spill_merge(
    wtd=np.zeros_like(topography, dtype=np.float64),
    topography=topography,
    ocean_level=ocean_level,
    nodata=np.nan,
    return_hierarchy=True,
)
print(dh.height, dh.width)

2048 512


In [5]:
def run_m(m):
    melt = topography * 0 + m
    return fill_spill_merge(wtd=melt, hierarchy=dh, nodata=np.nan)